# Notebook 05: Explainable AI using Grad-CAM

## Objective

This notebook applies Gradient-weighted Class Activation Mapping (Grad-CAM) to visualize the decision-making process of the trained EfficientNet-B2 model.

The notebook will:

- Load the trained EfficientNet-B2 model
- Generate Grad-CAM heatmaps
- Visualize model attention
- Compare Correct and Incorrect Predictions
- Save Grad-CAM images for research paper figures

Outputs will be stored inside the `results/gradcam/` directory.

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

from pathlib import Path
import os
import copy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import cv2

import torch
import torch.nn as nn

from torchvision import datasets, transforms

from torchvision.models import (
    efficientnet_b2,
    EfficientNet_B2_Weights
)

sns.set_theme(style="whitegrid")

print("="*60)
print("✅ Libraries Imported Successfully")
print("="*60)

✅ Libraries Imported Successfully


# Project Configuration

Configure all project paths and create the Grad-CAM output directory.

In [5]:
# ============================================================
# Project Configuration
# ============================================================

from pathlib import Path

# If notebook runs inside notebooks/
PROJECT_DIR = Path.cwd().parent

DATASET_DIR = PROJECT_DIR / "dataset" / "processed_dataset"

TEST_DIR = DATASET_DIR / "test"

MODEL_PATH = PROJECT_DIR / "models" / "best_model.pth"

RESULTS_DIR = PROJECT_DIR / "results"

GRADCAM_DIR = RESULTS_DIR / "gradcam"

GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("Project Directory :", PROJECT_DIR)
print("Dataset Directory :", DATASET_DIR)
print("Model Path        :", MODEL_PATH)
print("Results Directory :", GRADCAM_DIR)
print("Device            :", DEVICE)
print("=" * 60)

Project Directory : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI
Dataset Directory : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/dataset/processed_dataset
Model Path        : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/models/best_model.pth
Results Directory : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/results/gradcam
Device            : cuda


# Dataset Configuration

Configure image preprocessing and class information.

In [3]:
# ============================================================
# Dataset Configuration
# ============================================================

IMAGE_SIZE = 260

NUM_CLASSES = 3

CLASS_NAMES = [
    "BACTERIA",
    "NORMAL",
    "VIRUS"
]

transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

print("✅ Dataset configuration completed.")

✅ Dataset configuration completed.


# Load Trained EfficientNet-B2 Model

Load the trained EfficientNet-B2 model and prepare it for Grad-CAM visualization.

In [6]:
# ============================================================
# Load Trained EfficientNet-B2 Model
# ============================================================

weights = EfficientNet_B2_Weights.DEFAULT

model = efficientnet_b2(weights=weights)

in_features = model.classifier[1].in_features

# Must match Notebook 03 exactly
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()

print("=" * 60)
print("✅ EfficientNet-B2 Loaded Successfully")
print("=" * 60)
print(model.classifier)

✅ EfficientNet-B2 Loaded Successfully
Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Sequential(
    (0): Dropout(p=0.4, inplace=False)
    (1): Linear(in_features=1408, out_features=3, bias=True)
  )
)
